<a href="https://colab.research.google.com/github/dee431/ANIMATIONS/blob/main/Interactive_Nebula_Hand_Tracking_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **cell 1**

In [3]:
%%html
<!DOCTYPE html>
<html lang="en" class="h-full">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Interactive Nebula Hand Tracking Simulation</title>
    <script src="https://cdn.tailwindcss.com"></script>
    <script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/camera_utils/camera_utils.js" crossorigin="anonymous"></script>
    <script src="https://cdn.jsdelivr.net/npm/@mediapipe/hands/hands.js" crossorigin="anonymous"></script>
    <script src="https://unpkg.com/lucide@latest"></script>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&display=swap');
        body {
            font-family: 'Inter', sans-serif;
            background-color: #03030c;
            color: #f3f4f6;
            overflow: hidden;
            margin: 0;
            padding: 0;
        }
        ::-webkit-scrollbar {
            width: 6px;
        }
        ::-webkit-scrollbar-track {
            background: rgba(15, 23, 42, 0.6);
        }
        ::-webkit-scrollbar-thumb {
            background: rgba(139, 92, 246, 0.5);
            border-radius: 3px;
        }
        ::-webkit-scrollbar-thumb:hover {
            background: rgba(139, 92, 246, 0.8);
        }
        .glass-panel {
            background: rgba(15, 23, 42, 0.75);
            backdrop-filter: blur(16px);
            -webkit-backdrop-filter: blur(16px);
            border: 1px solid rgba(255, 255, 255, 0.08);
        }
        .glow-text {
            text-shadow: 0 0 20px rgba(168, 85, 247, 0.6), 0 0 40px rgba(236, 72, 153, 0.4);
        }
    </style>
</head>
<body class="h-full w-full relative select-none">

    <div id="canvas-container" class="absolute inset-0 z-0"></div>
    <video id="webcam" class="hidden" playsinline muted></video>

    <!-- UI Overlay -->
    <div class="relative z-10 h-full w-full pointer-events-none flex flex-col justify-between p-4 md:p-6">

        <!-- Top Navigation / Header -->
        <header class="flex flex-col sm:flex-row justify-between items-start sm:items-center gap-4 pointer-events-auto">
            <div class="glass-panel px-5 py-3 rounded-2xl shadow-2xl flex items-center gap-3 border border-purple-500/20">
                <div class="w-10 h-10 rounded-xl bg-gradient-to-tr from-purple-600 via-pink-600 to-amber-500 flex items-center justify-center shadow-lg shadow-purple-500/30">
                    <i data-lucide="sparkles" class="w-5 h-5 text-white animate-pulse"></i>
                </div>
                <div>
                    <h1 class="font-bold text-lg text-transparent bg-clip-text bg-gradient-to-r from-purple-300 via-pink-300 to-amber-200 glow-text">Stellar Nebula</h1>
                    <p class="text-xs text-purple-200/70 font-medium">Interactive Hand & Gesture Simulation</p>
                </div>
            </div>

            <!-- Status & Mode Badge -->
            <div class="flex items-center gap-2">
                <div id="tracking-status" class="glass-panel px-4 py-2.5 rounded-xl flex items-center gap-2 text-xs font-semibold shadow-lg transition-all duration-300 border border-amber-500/30">
                    <span id="status-dot" class="w-2.5 h-2.5 rounded-full bg-amber-400 animate-ping"></span>
                    <span id="status-text" class="text-amber-200">Initializing Camera...</span>
                </div>
                <button id="mode-toggle-btn" class="glass-panel px-4 py-2.5 rounded-xl hover:bg-purple-900/40 transition flex items-center gap-2 text-xs font-semibold shadow-lg text-purple-200 border border-purple-500/30 cursor-pointer">
                    <i data-lucide="video" class="w-4 h-4 text-purple-400"></i>
                    <span id="mode-btn-text">Using Mouse</span>
                </button>
            </div>
        </header>

        <!-- Center Guide / Hint -->
        <div id="guide-overlay" class="absolute top-1/2 left-1/2 -transform -translate-x-1/2 -translate-y-1/2 pointer-events-none transition-opacity duration-700 text-center">
            <div class="glass-panel px-6 py-4 rounded-3xl inline-flex flex-col items-center gap-2 border border-purple-500/30 shadow-2xl backdrop-blur-xl">
                <div class="flex items-center gap-3 text-purple-300">
                    <i data-lucide="hand" class="w-6 h-6 animate-bounce text-pink-400"></i>
                    <span class="text-sm font-semibold tracking-wide">Show your hand to the camera</span>
                </div>
                <p class="text-xs text-slate-400 max-w-xs">Move hand to steer nebula. Pinch thumb & index finger to scale size.</p>
                <span class="text-[10px] text-amber-300/80 bg-amber-950/40 px-2.5 py-1 rounded-full border border-amber-500/20 mt-1">Or drag mouse anywhere if camera is off</span>
            </div>
        </div>

        <!-- Bottom Controls & Instructions Panel -->
        <footer class="flex flex-col md:flex-row justify-between items-end md:items-center gap-4 pointer-events-auto">

            <!-- Instructions Card -->
            <div class="glass-panel p-4 rounded-2xl max-w-sm hidden sm:block shadow-2xl border border-purple-500/20">
                <div class="flex items-center gap-2 mb-2">
                    <i data-lucide="info" class="w-4 h-4 text-purple-400"></i>
                    <h3 class="text-xs font-bold uppercase tracking-wider text-purple-200">Gesture Controls</h3>
                </div>
                <ul class="text-xs text-slate-300 space-y-1.5">
                    <li class="flex items-center gap-2">
                        <span class="w-1.5 h-1.5 rounded-full bg-pink-400"></span>
                        <span><b>Hand Position:</b> Drifts nebula in 3D space</span>
                    </li>
                    <li class="flex items-center gap-2">
                        <span class="w-1.5 h-1.5 rounded-full bg-purple-400"></span>
                        <span><b>Pinch Distance:</b> Expands or shrinks nebula</span>
                    </li>
                    <li class="flex items-center gap-2">
                        <span class="w-1.5 h-1.5 rounded-full bg-amber-400"></span>
                        <span><b>Mouse Fallback:</b> Drag & Scroll wheel if no camera</span>
                    </li>
                </ul>
            </div>

            <!-- Interactive Settings Panel -->
            <div class="glass-panel p-4 rounded-2xl flex flex-wrap items-center gap-4 shadow-2xl border border-purple-500/20 w-full md:w-auto justify-between md:justify-start">
                <div class="flex items-center gap-3">
                    <label for="color-theme" class="text-xs font-semibold text-purple-200 flex items-center gap-1.5">
                        <i data-lucide="palette" class="w-4 h-4 text-pink-400"></i>
                        <span>Palette</span>
                    </label>
                    <select id="color-theme" class="bg-slate-900/80 border border-purple-500/30 text-purple-200 text-xs rounded-xl px-3 py-2 outline-none focus:border-purple-400 cursor-pointer">
                        <option value="cosmic">Cosmic Purple & Gold</option>
                        <option value="aurora">Emerald Aurora</option>
                        <option value="supernova">Supernova Crimson</option>
                        <option value="deepspace">Deep Cyan & Blue</option>
                    </select>
                </div>

                <div class="flex items-center gap-3">
                    <label for="particle-density" class="text-xs font-semibold text-purple-200 flex items-center gap-1.5">
                        <i data-lucide="layers" class="w-4 h-4 text-purple-400"></i>
                        <span>Density</span>
                    </label>
                    <input id="particle-density" type="range" min="5000" max="25000" step="1000" value="15000" class="accent-purple-500 cursor-pointer w-24 sm:w-32">
                </div>

                <div class="flex items-center gap-2">
                    <button id="reset-btn" class="bg-purple-600/60 hover:bg-purple-600 text-white p-2.5 rounded-xl transition shadow-lg shadow-purple-600/30 flex items-center gap-1.5 text-xs font-semibold cursor-pointer border border-purple-400/30" title="Reset View">
                        <i data-lucide="rotate-ccw" class="w-4 h-4"></i>
                        <span class="hidden sm:inline">Reset</span>
                    </button>
                    <button id="toggle-cam-btn" class="bg-gradient-to-r from-pink-600 to-purple-600 hover:from-pink-500 hover:to-purple-500 text-white px-3.5 py-2.5 rounded-xl transition shadow-lg shadow-pink-600/30 flex items-center gap-2 text-xs font-semibold cursor-pointer border border-pink-400/30">
                        <i data-lucide="camera" class="w-4 h-4"></i>
                        <span id="cam-btn-text">Toggle Camera</span>
                    </button>
                </div>
            </div>

        </footer>
    </div>

    <script>
        // Initialize Lucide icons
        lucide.createIcons();

        // Application State
        const state = {
            targetPosition: new THREE.Vector3(0, 0, 0),
            currentPosition: new THREE.Vector3(0, 0, 0),
            targetScale: 1.0,
            currentScale: 1.0,
            isCameraActive: false,
            useMouseFallback: true,
            theme: 'cosmic',
            particleCount: 15000,
            handDetected: false,
            lastHandTime: 0
        };

        // Scene Setup
        const container = document.getElementById('canvas-container');
        const scene = new THREE.Scene();
        scene.fog = new THREE.FogExp2(0x03030c, 0.015);

        const camera = new THREE.PerspectiveCamera(60, window.innerWidth / window.innerHeight, 0.1, 1000);
        camera.position.z = 35;

        const renderer = new THREE.WebGLRenderer({ antialias: true, alpha: false, powerPreference: "high-performance" });
        renderer.setSize(window.innerWidth, window.innerHeight);
        renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
        container.appendChild(renderer.domElement);

        // Lighting
        const ambientLight = new THREE.AmbientLight(0xffffff, 0.8);
        scene.add(ambientLight);

        const pointLight = new THREE.PointLight(0xa855f7, 3, 50);
        pointLight.position.set(0, 0, 10);
        scene.add(pointLight);

        let nebulaParticles = null;
        let starField = null;

        const colorPalettes = {
            cosmic: {
                colors: [new THREE.Color('#9333ea'), new THREE.Color('#ec4899'), new THREE.Color('#f59e0b'), new THREE.Color('#3b82f6')],
                coreColor: '#d946ef'
            },
            aurora: {
                colors: [new THREE.Color('#10b981'), new THREE.Color('#06b6d4'), new THREE.Color('#6366f1'), new THREE.Color('#a855f7')],
                coreColor: '#34d399'
            },
            supernova: {
                colors: [new THREE.Color('#ef4444'), new THREE.Color('#f97316'), new THREE.Color('#eab308'), new THREE.Color('#db2777')],
                coreColor: '#fb7185'
            },
            deepspace: {
                colors: [new THREE.Color('#3b82f6'), new THREE.Color('#0ea5e9'), new THREE.Color('#8b5cf6'), new THREE.Color('#047857')],
                coreColor: '#60a5fa'
            }
        };

        function createNebulaTexture() {
            const canvas = document.createElement('canvas');
            canvas.width = 128;
            canvas.height = 128;
            const ctx = canvas.getContext('2d');
            const gradient = ctx.createRadialGradient(64, 64, 0, 64, 64, 64);
            gradient.addColorStop(0, 'rgba(255, 255, 255, 1)');
            gradient.addColorStop(0.2, 'rgba(240, 200, 255, 0.8)');
            gradient.addColorStop(0.5, 'rgba(150, 50, 200, 0.2)');
            gradient.addColorStop(1, 'rgba(0, 0, 0, 0)');
            ctx.fillStyle = gradient;
            ctx.fillRect(0, 0, 128, 128);
            return new THREE.CanvasTexture(canvas);
        }

        const particleTexture = createNebulaTexture();

        function buildNebula() {
            if (nebulaParticles) scene.remove(nebulaParticles);
            if (starField) scene.remove(starField);

            const count = state.particleCount;
            const geometry = new THREE.BufferGeometry();
            const positions = new Float32Array(count * 3);
            const colors = new Float32Array(count * 3);
            const scales = new Float32Array(count);

            const palette = colorPalettes[state.theme];

            for (let i = 0; i < count; i++) {
                const u = Math.random();
                const radius = Math.pow(u, 0.5) * 18;
                const theta = Math.random() * Math.PI * 2 + radius * 0.3;

                const x = Math.cos(theta) * radius + (Math.random() - 0.5) * 4;
                const y = (Math.random() - 0.5) * (6 / (radius * 0.2 + 1));
                const z = Math.sin(theta) * radius + (Math.random() - 0.5) * 4;

                positions[i * 3] = x;
                positions[i * 3 + 1] = y;
                positions[i * 3 + 2] = z;

                const distRatio = radius / 18;
                const baseColor = palette.colors[Math.floor(Math.random() * palette.colors.length)];
                const mixedColor = baseColor.clone().lerp(new THREE.Color(palette.coreColor), 1 - distRatio);

                colors[i * 3] = mixedColor.r;
                colors[i * 3 + 1] = mixedColor.g;
                colors[i * 3 + 2] = mixedColor.b;

                scales[i] = Math.random() * 2.5 + 0.5;
            }

            geometry.setAttribute('position', new THREE.BufferAttribute(positions, 3));
            geometry.setAttribute('color', new THREE.BufferAttribute(colors, 3));
            geometry.setAttribute('size', new THREE.BufferAttribute(scales, 1));

            const material = new THREE.PointsMaterial({
                size: 0.8,
                vertexColors: true,
                map: particleTexture,
                transparent: true,
                opacity: 0.85,
                blending: THREE.AdditiveBlending,
                depthWrite: false
            });

            nebulaParticles = new THREE.Points(geometry, material);
            scene.add(nebulaParticles);

            const starCount = 3000;
            const starGeo = new THREE.BufferGeometry();
            const starPos = new Float32Array(starCount * 3);
            for (let i = 0; i < starCount * 3; i += 3) {
                starPos[i] = (Math.random() - 0.5) * 120;
                starPos[i + 1] = (Math.random() - 0.5) * 120;
                starPos[i + 2] = (Math.random() - 0.5) * 120;
            }
            starGeo.setAttribute('position', new THREE.BufferAttribute(starPos, 3));
            const starMat = new THREE.PointsMaterial({
                size: 0.3,
                color: 0xffffff,
                transparent: true,
                opacity: 0.6,
                blending: THREE.AdditiveBlending
            });
            starField = new THREE.Points(starGeo, starMat);
            scene.add(starField);
        }

        buildNebula();

        let mouse = { x: 0, y: 0, normalizedX: 0, normalizedY: 0, isDown: false };

        window.addEventListener('mousemove', (e) => {
            if (!state.useMouseFallback && state.handDetected) return;
            mouse.x = e.clientX;
            mouse.y = e.clientY;
            mouse.normalizedX = (e.clientX / window.innerWidth) * 2 - 1;
            mouse.normalizedY = -(e.clientY / window.innerHeight) * 2 + 1;

            if (mouse.isDown) {
                state.targetPosition.x = mouse.normalizedX * 12;
                state.targetPosition.y = mouse.normalizedY * 8;
            }
        });

        window.addEventListener('mousedown', () => { mouse.isDown = true; });
        window.addEventListener('mouseup', () => { mouse.isDown = false; });

        window.addEventListener('wheel', (e) => {
            if (!state.useMouseFallback && state.handDetected) return;
            state.targetScale = Math.max(0.4, Math.min(2.8, state.targetScale - e.deltaY * 0.002));
        });

        window.addEventListener('touchmove', (e) => {
            if (e.touches.length > 0) {
                const touch = e.touches[0];
                const nx = (touch.clientX / window.innerWidth) * 2 - 1;
                const ny = -(touch.clientY / window.innerHeight) * 2 + 1;
                state.targetPosition.x = nx * 10;
                state.targetPosition.y = ny * 6;
            }
        }, { passive: true });

        const videoElement = document.getElementById('webcam');
        const statusDot = document.getElementById('status-dot');
        const statusText = document.getElementById('status-text');
        const trackingStatus = document.getElementById('tracking-status');
        const guideOverlay = document.getElementById('guide-overlay');

        let handsInstance = null;
        let cameraUtilsInstance = null;

        function updateStatus(status, message, colorClass, dotClass) {
            statusText.textContent = message;
            trackingStatus.className = `glass-panel px-4 py-2.5 rounded-xl flex items-center gap-2 text-xs font-semibold shadow-lg transition-all duration-300 border ${colorClass}`;
            statusDot.className = `w-2.5 h-2.5 rounded-full ${dotClass}`;
        }

        function onResults(results) {
            const now = performance.now();
            if (results.multiHandLandmarks && results.multiHandLandmarks.length > 0) {
                state.handDetected = true;
                state.lastHandTime = now;
                state.useMouseFallback = false;

                document.getElementById('mode-btn-text').textContent = "Hand Tracking Active";
                guideOverlay.classList.add('opacity-0');
                updateStatus('tracking', 'Hand Tracked ✨', 'border-emerald-500/30 bg-emerald-950/20', 'bg-emerald-400 animate-pulse');

                const landmarks = results.multiHandLandmarks[0];
                const palmX = landmarks[9].x;
                const palmY = landmarks[9].y;

                state.targetPosition.x = ( (1 - palmX) - 0.5 ) * 20;
                state.targetPosition.y = ( -(palmY - 0.5) ) * 14;

                const thumbTip = landmarks[4];
                const indexTip = landmarks[8];
                const pinchDistance = Math.hypot(thumbTip.x - indexTip.x, thumbTip.y - indexTip.y, thumbTip.z - indexTip.z);

                const mappedScale = THREE.MathUtils.lerp(0.5, 2.5, Math.max(0, Math.min(1, (pinchDistance - 0.04) * 4)));
                state.targetScale = mappedScale;

            } else {
                if (now - state.lastHandTime > 1500 && !state.useMouseFallback) {
                    state.handDetected = false;
                    guideOverlay.classList.remove('opacity-0');
                    updateStatus('searching', 'Searching for Hand...', 'border-amber-500/30 bg-amber-950/20', 'bg-amber-400 animate-ping');
                }
            }
        }

        async function initHandTracking() {
            try {
                handsInstance = new Hands({
                    locateFile: (file) => `https://cdn.jsdelivr.net/npm/@mediapipe/hands/${file}`
                });

                handsInstance.setOptions({
                    maxNumHands: 1,
                    modelComplexity: 1,
                    minDetectionConfidence: 0.65,
                    minTrackingConfidence: 0.65
                });

                handsInstance.onResults(onResults);

                cameraUtilsInstance = new Camera(videoElement, {
                    onFrame: async () => {
                        if (state.isCameraActive) {
                            await handsInstance.send({ image: videoElement });
                        }
                    },
                    width: 640,
                    height: 480
                });

                await cameraUtilsInstance.start();
                state.isCameraActive = true;
                state.useMouseFallback = false;
                updateStatus('active', 'Camera Active', 'border-emerald-500/30 bg-emerald-950/20', 'bg-emerald-400');
            } catch (err) {
                console.warn("Camera access denied or unavailable, falling back to mouse interaction.", err);
                state.isCameraActive = false;
                state.useMouseFallback = true;
                updateStatus('fallback', 'Using Mouse Mode', 'border-purple-500/30 bg-purple-950/20', 'bg-purple-400');
                document.getElementById('mode-btn-text').textContent = "Using Mouse";
            }
        }

        const clock = new THREE.Clock();

        function animate() {
            requestAnimationFrame(animate);

            const elapsedTime = clock.getElapsedTime();

            state.currentPosition.lerp(state.targetPosition, 0.08);
            state.currentScale = THREE.MathUtils.lerp(state.currentScale, state.targetScale, 0.08);

            if (nebulaParticles) {
                nebulaParticles.position.copy(state.currentPosition);
                nebulaParticles.scale.set(state.currentScale, state.currentScale, state.currentScale);
                nebulaParticles.rotation.y = elapsedTime * 0.05;
                nebulaParticles.rotation.z = Math.sin(elapsedTime * 0.03) * 0.1;
            }

            if (starField) {
                starField.rotation.y = elapsedTime * 0.01;
            }

            pointLight.position.x = state.currentPosition.x;
            pointLight.position.y = state.currentPosition.y;

            renderer.render(scene, camera);
        }

        animate();

        document.getElementById('color-theme').addEventListener('change', (e) => {
            state.theme = e.target.value;
            buildNebula();
        });

        document.getElementById('particle-density').addEventListener('input', (e) => {
            state.particleCount = parseInt(e.target.value);
            buildNebula();
        });

        document.getElementById('reset-btn').addEventListener('click', () => {
            state.targetPosition.set(0, 0, 0);
            state.targetScale = 1.0;
        });

        const toggleCamBtn = document.getElementById('toggle-cam-btn');
        const modeToggleBtn = document.getElementById('mode-toggle-btn');

        toggleCamBtn.addEventListener('click', async () => {
            if (!state.isCameraActive) {
                toggleCamBtn.innerHTML = `<i data-lucide="loader" class="w-4 h-4 animate-spin"></i><span>Starting...</span>`;
                lucide.createIcons();
                await initHandTracking();
                toggleCamBtn.innerHTML = `<i data-lucide="video-off" class="w-4 h-4"></i><span>Stop Camera</span>`;
                lucide.createIcons();
            } else {
                state.isCameraActive = false;
                state.useMouseFallback = true;
                if (videoElement && videoElement.srcObject) {
                    const stream = videoElement.srcObject;
                    stream.getTracks().forEach(track => track.stop());
                    videoElement.srcObject = null;
                }
                updateStatus('fallback', 'Mouse Mode Active', 'border-purple-500/30 bg-purple-950/20', 'bg-purple-400');
                document.getElementById('mode-btn-text').textContent = "Using Mouse";
                toggleCamBtn.innerHTML = `<i data-lucide="camera" class="w-4 h-4"></i><span>Start Camera</span>`;
                guideOverlay.classList.remove('opacity-0');
                lucide.createIcons();
            }
        });

        modeToggleBtn.addEventListener('click', () => {
            state.useMouseFallback = !state.useMouseFallback;
            if (state.useMouseFallback) {
                document.getElementById('mode-btn-text').textContent = "Mouse Fallback";
                updateStatus('fallback', 'Mouse Control Mode', 'border-purple-500/30 bg-purple-950/20', 'bg-purple-400');
            } else if (state.isCameraActive) {
                document.getElementById('mode-btn-text').textContent = "Hand Tracking Active";
                updateStatus('tracking', 'Hand Tracked ✨', 'border-emerald-500/30 bg-emerald-950/20', 'bg-emerald-400');
            }
        });

        window.addEventListener('resize', () => {
            camera.aspect = window.innerWidth / window.innerHeight;
            camera.updateProjectionMatrix();
            renderer.setSize(window.innerWidth, window.innerHeight);
        });

        window.addEventListener('load', () => {
            initHandTracking();
        });
    </script>
</body>
</html>